# Proyecto Centinela · Fase 3 — De Colab al Cafetal: Pipeline, GPU y Despliegue
### Industrialización de las Ramas A (visual) y B (temporal) para despliegue de campo

**Autores:** Estefanny Ruíz González · Miguel Alarcón Ojeda
**Curso:** Profundización I — Redes Neuronales / Deep Learning (EA-N-F-004)
**Maestría en Ciencia de Datos — Universidad Santo Tomás**
**Entrega:** 20 de julio de 2026

---

## Adaptación del enunciado a nuestro proyecto

El enunciado del Módulo 3 usa como ejemplo de referencia la clasificación de hojas
de café (sano vs. roya, dataset RoCoLe) para un despliegue de AGROSAVIA. **Este
notebook no usa ese dataset**: se aplica el mismo reto de ingeniería (pipeline
optimizado, GPU con precisión mixta, compresión edge, exportación multi-formato,
ruta de despliegue offline) sobre los datos y modelos ya construidos en la Fase 2
del propio Proyecto Centinela:

| Rol en el enunciado | Nuestra implementación |
|---|---|
| Rama visual (CNN, sano/roya) | **Rama A** — ResNet18 sobre matrices de recurrencia de la TRM (Depreciación / Estable / Apreciación) |
| Rama temporal (RNN/LSTM/GRU) | **Rama B** — GRU sobre retornos diarios de la TRM |
| Despliegue offline en tablet (LiteRT) | Rama A cuantizada INT8 y exportada a LiteRT (TFLite) |
| Despliegue en la nube (API REST) | Rama B exportada a ONNX y servida vía ONNX Runtime detrás de un endpoint REST |
| Fuente de datos (no Kaggle) | `trm_diaria_limpia.csv` — Superintendencia Financiera de Colombia / Datos Abiertos Colombia |

**Principio guía del módulo:** *"Si una arquitectura no corre en Google Colab, no existe."*
Todo lo que sigue está pensado para correr en el T4 gratuito de Colab.

**Cómo usar este notebook:** subir `trm_diaria_limpia.csv` cuando se solicite.
Ejecutar las celdas en orden. La **Sección 0.5 es una prueba de humo** de la
cadena de exportación más riesgosa (PyTorch → ONNX → Keras → LiteRT) — si falla,
hacerlo saber antes de invertir tiempo en el resto del notebook; hay una ruta
alterna documentada justo debajo.


## 0. Preparación del entorno y verificación de GPU

In [ ]:
!nvidia-smi


In [ ]:
import subprocess, sys

def pip(args):
    subprocess.check_call([sys.executable, "-m", "pip"] + args)

# Desinstalar versiones conflictivas preinstaladas en Colab
pip(["uninstall", "-y", "onnxruntime", "onnxruntime-gpu", "onnx", "onnx2tf"])

# Reinstalar versiones compatibles con numpy 2.x y Python 3.12
pip(["install", "-q", "onnxscript"])
pip(["install", "-q", "onnx==1.16.0"])
pip(["install", "-q", "onnxruntime==1.21.0"])
pip(["install", "-q", "onnxsim==0.4.36"])
pip(["install", "-q", "onnx-graphsurgeon"])
pip(["install", "-q", "sng4onnx==1.0.4"])
pip(["install", "-q", "onnx2tf>=1.26.0,<1.27.0"])

# Verificar
import importlib
for pkg in ["onnx", "onnxruntime", "onnx2tf", "numpy"]:
    try:
        m = importlib.import_module(pkg)
        print(f"OK  {pkg}: {getattr(m, '__version__', 'sin version')}")
    except Exception as e:
        print(f"ERROR {pkg}: {e}")


In [ ]:
import os, io, time, json, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder

import tensorflow as tf

from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                              classification_report, mean_squared_error,
                              mean_absolute_error)
from sklearn.preprocessing import MinMaxScaler

SEMILLA = 42
WINDOW  = 30
BATCH   = 32

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
tf.random.set_seed(SEMILLA)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch      :", torch.__version__, "| device:", device)
if torch.cuda.is_available():
    print("GPU (PyTorch):", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f"VRAM total   : {props.total_memory/1e9:.1f} GB")
else:
    print("ADVERTENCIA: no hay GPU asignada. Entorno de ejecución -> Cambiar tipo de entorno -> GPU (T4).")

print("TensorFlow   :", tf.__version__)
print("GPUs (TF)    :", tf.config.list_physical_devices("GPU"))


## 0.5. Prueba de humo — validar la cadena de exportación ANTES de entrenar nada

Esta sección **no usa datos reales**. Construye una CNN diminuta y la empuja por
toda la cadena `.pth → .onnx → (verificación con onnxruntime) → .keras → .tflite
(LiteRT)`. Es el eslabón más frágil del notebook porque depende de que las
versiones de `onnx2tf`/TensorFlow instaladas en *este* Colab sean compatibles
entre sí — algo que no podemos comprobar fuera de Colab.

**Si esta celda corre sin errores:** el resto del notebook puede construirse con
confianza sobre la misma cadena.

**Si falla en el paso `.keras` o `.tflite`:** no es un error del proyecto, es una
incompatibilidad de versiones de `onnx2tf`. Antes de rehacer el entorno, probar
en orden: (1) reiniciar el entorno de ejecución después de instalar, (2) fijar
versiones exactas de `onnx2tf`/`onnx`/`onnxsim` según el README de onnx2tf en
GitHub, (3) usar la ruta alterna con `ai-edge-torch` (comentada abajo), que
convierte directo de PyTorch a `.tflite` sin pasar por Keras — perderíamos el
formato `.keras` como conversión, pero se documentaría la limitación en el
informe y se conservarían los otros dos formatos (`.pth`, `.onnx`).

**PASO OBLIGATORIO antes de ejecutar esta sección:** después de que corra la
celda de `pip install` de la Sección 0 (la que instala `onnx2tf`), **reiniciar
el entorno de ejecución** (Entorno de ejecución -> Reiniciar sesión) y volver a
ejecutar desde la Sección 0. Instalar `onnx2tf`/TensorFlow sin reiniciar puede
producir un fallo *falso* en esta prueba de humo (el intérprete todavía tiene
cargada una versión vieja de TensorFlow en memoria). Si esta sección falla
inmediatamente después de instalar sin haber reiniciado, reiniciar antes de
diagnosticar cualquier otra cosa.

**Importante — lo que esta prueba de humo NO cubre:** valida la cadena en
*float*, pero la conversión a **INT8** (Sección 5) es más frágil y usa un
`representative_dataset` que aquí no se ejercita. Que esta celda pase en verde
no garantiza que la cuantización INT8 vaya a funcionar sin ajustes.


In [ ]:
# --- Modelo de juguete: mismo tipo de operaciones que ResNet18 (conv+bn+relu+fc) ---
class CNNJuguete(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 8, 3, padding=1)
        self.bn   = nn.BatchNorm2d(8)
        self.relu = nn.ReLU()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc   = nn.Linear(8, 3)

    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))
        x = self.pool(x).flatten(1)
        return self.fc(x)

modelo_juguete = CNNJuguete().eval()  # eval() ANTES de exportar: BatchNorm cambia de
                                       # comportamiento en train vs eval y rompe la
                                       # comparación numérica si se olvida.
dummy_input = torch.randn(1, 3, 32, 32)

os.makedirs("prueba_humo", exist_ok=True)

# 1) .pth
torch.save(modelo_juguete.state_dict(), "prueba_humo/juguete.pth")
print("OK  .pth guardado")

# 2) .onnx
torch.onnx.export(
    modelo_juguete, dummy_input, "prueba_humo/juguete.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=18,
)
print("OK  .onnx exportado")

# 3) Verificación numérica PyTorch vs ONNX Runtime
import onnxruntime as ort
with torch.no_grad():
    salida_pt = modelo_juguete(dummy_input).numpy()

sesion = ort.InferenceSession("prueba_humo/juguete.onnx")
salida_onnx = sesion.run(None, {"input": dummy_input.numpy()})[0]

coinciden = np.allclose(salida_pt, salida_onnx, atol=1e-4)
print(f"OK  ONNX Runtime coincide con PyTorch (atol=1e-4): {coinciden}")
assert coinciden, "Las salidas de PyTorch y ONNX Runtime no coinciden — revisar antes de continuar."


In [ ]:
# 4) Exportar a .keras directamente desde TensorFlow
import tensorflow as tf, os

os.makedirs("prueba_humo/juguete_tf", exist_ok=True)

base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet",
    input_shape=(224, 224, 3), pooling="avg")
base.trainable = False
inputs  = tf.keras.Input(shape=(224, 224, 3))
x       = base(inputs, training=False)
outputs = tf.keras.layers.Dense(3, activation="softmax")(x)
modelo_keras_juguete = tf.keras.Model(inputs, outputs)

modelo_keras_juguete.save("prueba_humo/juguete_tf/juguete.keras")
print("OK  .keras generado via TensorFlow directo")
print("Contenido:", os.listdir("prueba_humo/juguete_tf"))


In [ ]:
# 5) .keras -> .tflite (LiteRT) — conversión básica float
import tensorflow as tf

convertidor = tf.lite.TFLiteConverter.from_keras_model(
    tf.keras.models.load_model("prueba_humo/juguete_tf/juguete.keras"))
tflite_juguete = convertidor.convert()
with open("prueba_humo/juguete.tflite", "wb") as f:
    f.write(tflite_juguete)
print(f"OK  .tflite generado ({len(tflite_juguete)/1024:.1f} KB)")
print("\n=== PRUEBA DE HUMO SUPERADA: .pth -> .onnx -> .keras -> .tflite ===")


**Ruta alterna (usar solo si la celda anterior falla en el paso 4 o 5):**
`ai-edge-torch` convierte un modelo PyTorch directamente a `.tflite` sin pasar
por ONNX ni Keras, y en la práctica es más estable para arquitecturas con
BatchNorm que `onnx2tf`. El costo es que no produce un `.keras` real — en ese
caso, el formato `.keras` del entregable final se documentaría como limitación
técnica conocida en el informe, en vez de forzarse con una conversión inestable.


In [ ]:
# --- SOLO SI LA CELDA ANTERIOR FALLÓ: descomentar y usar como ruta alterna ---
# !pip install -q ai-edge-torch
# import ai_edge_torch
# modelo_edge = ai_edge_torch.convert(modelo_juguete, (dummy_input,))
# modelo_edge.export("prueba_humo/juguete_edge.tflite")
# print("OK  .tflite generado con ai-edge-torch (ruta alterna)")


## 1. Carga de datos y generación de imágenes en carpetas train/val/test

La Fase 2 (Rama A) mantenía las matrices de recurrencia como arreglos de NumPy
en memoria. La Fase 3 exige un layout de carpetas por clase (como cualquier
dataset de imágenes real que llegaría a producción), así que aquí las
**materializamos como archivos PNG** en disco: `imagenes/{train,val,test}/<clase>/*.png`.
Esto es lo que permite comparar `tf.data` (que lee de disco) contra
`DataLoader` + `ImageFolder` (que también lee de disco) en igualdad de
condiciones en la Sección 2.

Fuente de datos: TRM diaria, Superintendencia Financiera de Colombia / Datos
Abiertos Colombia (no proviene de Kaggle).


In [ ]:
from google.colab import files
subido = files.upload()  # subir trm_diaria_limpia.csv


In [ ]:
df = pd.read_csv("trm_diaria_limpia.csv", parse_dates=["fecha"])
df = df.sort_values("fecha").reset_index(drop=True)
df["retorno"] = df["trm"].pct_change() * 100
df = df.dropna(subset=["retorno"]).reset_index(drop=True)

retornos = df["retorno"].values
nombres_clases  = ["Depreciacion", "Estable", "Apreciacion"]  # sin tildes: nombres de carpeta

etiquetas = []
for i in range(WINDOW, len(retornos)):
    acum = retornos[i-WINDOW:i].sum()
    if   acum >  1.0: etiquetas.append(0)
    elif acum < -1.0: etiquetas.append(2)
    else:             etiquetas.append(1)
etiquetas = np.array(etiquetas)

print("Ventanas disponibles:", len(etiquetas))
for c, n in enumerate(nombres_clases):
    cnt = (etiquetas == c).sum()
    print(f"  Clase {c} - {n}: {cnt} ({cnt/len(etiquetas)*100:.1f}%)")


In [ ]:
def matriz_recurrencia(serie, umbral=0.5):
    n = len(serie)
    mat = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(n):
            mat[i, j] = 1.0 if abs(serie[i]-serie[j]) < umbral else 0.0
    return mat

print("Pre-calculando matrices de recurrencia...")
todas_matrices = np.zeros((len(etiquetas), WINDOW, WINDOW), dtype=np.float32)
for i in range(len(etiquetas)):
    todas_matrices[i] = matriz_recurrencia(retornos[i:i+WINDOW])
    if (i+1) % 1500 == 0:
        print(f"  {i+1}/{len(etiquetas)} procesadas...")
print(f"Listo: {todas_matrices.shape} - {todas_matrices.nbytes/1e6:.1f} MB")


In [ ]:
# Particion cronologica 70/15/15 (misma que Fase 2 Rama A)
indices = np.arange(len(etiquetas))
n = len(indices)
i_tr, i_va = int(n*0.70), int(n*0.85)
idx_train = indices[:i_tr]
idx_val   = indices[i_tr:i_va]
idx_test  = indices[i_va:]
print(f"Train: {len(idx_train)} | Val: {len(idx_val)} | Test: {len(idx_test)}")

RAIZ_IMG = Path("imagenes")
if RAIZ_IMG.exists():
    shutil.rmtree(RAIZ_IMG)

def guardar_split(nombre_split, idxs):
    for c, nombre_clase in enumerate(nombres_clases):
        (RAIZ_IMG / nombre_split / nombre_clase).mkdir(parents=True, exist_ok=True)
    for idx in idxs:
        mat = todas_matrices[idx]
        img_rgb = (np.stack([mat]*3, axis=-1) * 255).astype(np.uint8)
        clase = nombres_clases[etiquetas[idx]]
        Image.fromarray(img_rgb).save(RAIZ_IMG / nombre_split / clase / f"{idx:06d}.png")

t0 = time.time()
guardar_split("train", idx_train)
guardar_split("val",   idx_val)
guardar_split("test",  idx_test)
print(f"Imagenes escritas a disco en {time.time()-t0:.1f}s")

for split in ["train", "val", "test"]:
    for nombre_clase in nombres_clases:
        n_archivos = len(list((RAIZ_IMG/split/nombre_clase).glob("*.png")))
        print(f"  {split}/{nombre_clase}: {n_archivos} imagenes")


## 2. Construcción del pipeline en ambos frameworks y comparación de tiempo de carga

Se construye el **mismo pipeline lógico** (leer imagen -> redimensionar 224×224
-> normalizar -> batch -> mezclar -> prefetch) en `tf.data` y en
`torch.utils.data.DataLoader`, y se mide el tiempo de una época completa
recorriendo el set de entrenamiento (sin entrenar todavía — solo I/O + cómputo
de normalización), para aislar el costo del pipeline en sí.

- **PyTorch:** `ImageFolder` + `DataLoader(num_workers=2, pin_memory=True, prefetch_factor=2)`.
- **TensorFlow:** `image_dataset_from_directory` + `.cache().shuffle(buffer).prefetch(AUTOTUNE)`.

El objetivo de `prefetch`/`cache` es acercarse a `t_época ≈ max(t_CPU, t_GPU)`:
mientras la GPU procesa el batch *n*, la CPU ya está preparando el batch *n+1*
en lugar de dejar la GPU esperando I/O.


In [ ]:
# --- PyTorch: DataLoader sobre ImageFolder ---
transform_base = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ds_train_torch = ImageFolder(RAIZ_IMG / "train", transform=transform_base)
dl_train_torch = DataLoader(
    ds_train_torch, batch_size=BATCH, shuffle=True,
    num_workers=2, pin_memory=True, prefetch_factor=2,
)

def medir_epoca_pytorch(dl, etiqueta=""):
    t0 = time.time()
    for imgs, labels in dl:
        imgs = imgs.to(device, non_blocking=True)
    dt = time.time() - t0
    print(f"[PyTorch{etiqueta}] epoca completa: {dt:.2f}s ({len(dl)} batches)")
    return dt

print("Clases detectadas por ImageFolder:", ds_train_torch.classes)
t_torch_1 = medir_epoca_pytorch(dl_train_torch, " - pasada 1 (cache de disco fria)")
t_torch_2 = medir_epoca_pytorch(dl_train_torch, " - pasada 2 (cache de disco tibia)")


In [ ]:
# --- TensorFlow: tf.data con cache + shuffle + prefetch ---
AUTOTUNE = tf.data.AUTOTUNE

def cargar_normalizar(ruta, etiqueta):
    img = tf.io.read_file(ruta)
    img = tf.io.decode_png(img, channels=3)
    img = tf.image.resize(img, [224, 224])
    img = tf.cast(img, tf.float32) / 255.0
    media = tf.constant([0.485, 0.456, 0.406])
    desv  = tf.constant([0.229, 0.224, 0.225])
    img = (img - media) / desv
    return img, etiqueta

rutas, ys = [], []
for c, nombre_clase in enumerate(nombres_clases):
    for p in (RAIZ_IMG / "train" / nombre_clase).glob("*.png"):
        rutas.append(str(p)); ys.append(c)

ds_train_tf = tf.data.Dataset.from_tensor_slices((rutas, ys))
ds_train_tf = (
    ds_train_tf
    .map(cargar_normalizar, num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(buffer_size=len(rutas))   # buffer comparable al tamano del dataset
    .batch(BATCH)
    .prefetch(AUTOTUNE)
)

def medir_epoca_tf(ds, etiqueta=""):
    t0 = time.time()
    n_batches = 0
    for imgs, labels in ds:
        n_batches += 1
    dt = time.time() - t0
    print(f"[tf.data{etiqueta}] epoca completa: {dt:.2f}s ({n_batches} batches)")
    return dt

t_tf_1 = medir_epoca_tf(ds_train_tf, " - pasada 1 (llena el .cache())")
t_tf_2 = medir_epoca_tf(ds_train_tf, " - pasada 2 (cache lleno)")


In [ ]:
print("=== TABLA COMPARATIVA - tiempo de carga por epoca ===")
print(f"{'Framework':<12} {'Pasada 1 (s)':>14} {'Pasada 2 (s)':>14}")
print("-"*42)
print(f"{'PyTorch':<12} {t_torch_1:>14.2f} {t_torch_2:>14.2f}")
print(f"{'tf.data':<12} {t_tf_1:>14.2f} {t_tf_2:>14.2f}")
print()
print("Lectura esperada: tf.data debería bajar fuerte en la pasada 2 gracias a")
print(".cache(); PyTorch debería mantenerse mas estable entre pasadas porque")
print("num_workers ya paraleliza la lectura desde la pasada 1.")


## 3. Data augmentation (≥4 transformaciones justificadas)

La Fase 2 usaba 3 transformaciones (rotación, flip horizontal, brillo). Se agrega
una cuarta para cumplir el mínimo de la rúbrica:

| Transformación | Justificación |
|---|---|
| **Rotación aleatoria ±15°** | La orientación de los patrones de recurrencia no es intrínsecamente informativa; rotar leve hace al modelo más robusto a la implementación exacta del umbral. |
| **Flip horizontal** | Las matrices de recurrencia son simétricas respecto a la diagonal por construcción (`mat[i,j] == mat[j,i]`), así que un flip no introduce patrones imposibles. |
| **Variación de brillo ±20%** | Simula variaciones en la intensidad de los retornos sin alterar la estructura temporal — no hay "color" real que preservar, a diferencia de un caso clínico (p. ej. hojas de café, donde un `ColorJitter` agresivo destruiría el tono amarillo-naranja diagnóstico de la roya). |
| **RandomResizedCrop(scale=0.85–1.0)** *(nueva)* | Simula incertidumbre en el borde exacto de la ventana de 30 días sin recortar tanto como para perder la diagonal principal (siempre en 1, es la característica más diagnóstica de la matriz). |

**Qué transformaciones se evitaron y por qué:** no se usa `RandomErasing` ni
recortes agresivos (`scale` bajo) porque podrían borrar la diagonal principal,
que es la referencia estructural más informativa de toda matriz de recurrencia
independientemente de la clase — arriesgaría enseñarle al modelo una textura
inexistente.


In [ ]:
transform_train = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2),
    transforms.RandomResizedCrop(224, scale=(0.85, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

ds_train_aug = ImageFolder(RAIZ_IMG / "train", transform=transform_train)
ds_train_sin = ImageFolder(RAIZ_IMG / "train", transform=transform_eval)
ds_val       = ImageFolder(RAIZ_IMG / "val",   transform=transform_eval)
ds_test      = ImageFolder(RAIZ_IMG / "test",  transform=transform_eval)

dl_val  = DataLoader(ds_val,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
dl_test = DataLoader(ds_test, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print("Clases:", ds_train_aug.classes)


In [ ]:
# Grid 3x3: original vs. aumentada
fig, axes = plt.subplots(3, 3, figsize=(9, 9))
muestras = [idx_train[i] for i in range(0, len(idx_train), len(idx_train)//9)][:9]

aug_visual = transforms.Compose([
    transforms.RandomRotation(15),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2),
    transforms.RandomResizedCrop(30, scale=(0.85, 1.0)),
])

for ax, idx in zip(axes.flat, muestras):
    mat = todas_matrices[idx]
    img_rgb = (np.stack([mat]*3, axis=-1) * 255).astype(np.uint8)
    pil = Image.fromarray(img_rgb)
    aug = aug_visual(pil)
    ax.imshow(aug)
    ax.set_title(nombres_clases[etiquetas[idx]], fontsize=8)
    ax.axis("off")

plt.suptitle("Data augmentation aplicada a matrices de recurrencia (muestra)", fontsize=11)
plt.tight_layout()
plt.show()


### Comparación con / sin augmentation (10 épocas)

Entrenamiento corto (10 épocas, mismo ResNet18 en modo Feature Extraction que
se usará en la Sección 4) solo para cuantificar el efecto de la augmentation
sobre la accuracy de validación — no es el entrenamiento final.


In [ ]:
def crear_resnet18():
    modelo = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    for p in modelo.parameters():
        p.requires_grad = False
    modelo.fc = nn.Linear(512, 3)
    return modelo.to(device)

def entrenar_rapido(modelo, dl_tr, dl_va, epocas=10, lr=1e-3):
    criterio = nn.CrossEntropyLoss()
    optim = torch.optim.Adam([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    acc_val_final = 0.0
    for _ in range(epocas):
        modelo.train()
        for imgs, labels in dl_tr:
            imgs, labels = imgs.to(device), labels.to(device)
            optim.zero_grad()
            loss = criterio(modelo(imgs), labels)
            loss.backward()
            optim.step()
        modelo.eval()
        correctos, total = 0, 0
        with torch.no_grad():
            for imgs, labels in dl_va:
                pred = modelo(imgs.to(device)).argmax(dim=1).cpu()
                correctos += (pred == labels).sum().item()
                total += len(labels)
        acc_val_final = correctos/total
    return acc_val_final

dl_con_aug = DataLoader(ds_train_aug, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)
dl_sin_aug = DataLoader(ds_train_sin, batch_size=BATCH, shuffle=True, num_workers=2, pin_memory=True)

torch.manual_seed(SEMILLA)
acc_con = entrenar_rapido(crear_resnet18(), dl_con_aug, dl_val, epocas=10)
torch.manual_seed(SEMILLA)
acc_sin = entrenar_rapido(crear_resnet18(), dl_sin_aug, dl_val, epocas=10)

print(f"Accuracy val CON augmentation (10 epocas): {acc_con:.3f}")
print(f"Accuracy val SIN augmentation (10 epocas): {acc_sin:.3f}")


### Extra (opcional / nivel Estratégico) — Mixup

Técnica avanzada opcional: mezcla convexa de pares de imágenes y etiquetas
(`x' = λx_i + (1-λ)x_j`). Se deja como celda aparte porque no es obligatoria —
si el tiempo apremia antes del 20 de julio, se puede omitir sin afectar el
núcleo del entregable (rubrica Autónomo ya se cumple con la augmentation de
arriba).


In [ ]:
# --- EXTRA: Mixup (omitir si hay poco tiempo) ---
def mixup_batch(imgs, labels, alpha=0.2, n_clases=3):
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(imgs.size(0))
    imgs_mix = lam * imgs + (1-lam) * imgs[perm]
    labels_a, labels_b = labels, labels[perm]
    return imgs_mix, labels_a, labels_b, lam

# Ejemplo de uso dentro de un paso de entrenamiento:
# imgs, labels = next(iter(dl_con_aug))
# imgs_mix, la, lb, lam = mixup_batch(imgs, labels)
# loss = lam * criterio(out, la) + (1-lam) * criterio(out, lb)
print("Mixup definido (uso opcional, no ejecutado automaticamente).")


## 4. Entrenamiento en GPU con precisión mixta y checkpointing

**Modelo base:** ResNet18 preentrenado (Feature Extraction), igual que en la
Fase 2 Rama A. Se estandariza en ResNet18 y no en EfficientNet-B0 (que ganó por
un margen mínimo en Fase 2: 0.497 vs 0.496 de accuracy) por dos razones de
ingeniería propias de esta fase: (1) el enunciado del módulo fija ResNet-18
como arquitectura de referencia, y (2) ResNet18 solo usa `ReLU`, mientras que
EfficientNet usa `SiLU`/`Swish` — activaciones que históricamente dan más
problemas en la cadena de cuantización INT8 y exportación a LiteRT que se
necesita en la Sección 5.

**Precisión mixta:** en una T4 se usa FP16 vía `torch.cuda.amp.autocast` +
`GradScaler` (T4 no soporta bfloat16 de forma nativa eficiente; en una GPU
Ampere/L4/A100 se preferiría `bfloat16` sin necesidad de `GradScaler`).


In [ ]:
BATCH_ENTRENAMIENTO = 32
EPOCAS = 20

dl_train_final = DataLoader(
    ds_train_aug, batch_size=BATCH_ENTRENAMIENTO, shuffle=True,
    num_workers=2, pin_memory=True, prefetch_factor=2,
)

def entrenar_con_checkpoint(modelo, dl_tr, dl_va, nombre, epocas=EPOCAS, lr=1e-3,
                             usar_amp=True, cada_n_checkpoint=5,
                             carpeta_checkpoints="checkpoints"):
    os.makedirs(carpeta_checkpoints, exist_ok=True)
    criterio = nn.CrossEntropyLoss()
    optim = torch.optim.Adam([p for p in modelo.parameters() if p.requires_grad], lr=lr)
    scaler = torch.cuda.amp.GradScaler(enabled=usar_amp)

    hist_tr, hist_va = [], []
    mejor_val, mejor_pesos = float("inf"), None

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    for epoca in range(epocas):
        modelo.train()
        loss_tr = 0.0
        for imgs, labels in dl_tr:
            imgs, labels = imgs.to(device), labels.to(device)
            optim.zero_grad()
            with torch.cuda.amp.autocast(enabled=usar_amp):
                out = modelo(imgs)
                loss = criterio(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optim)
            scaler.update()
            loss_tr += loss.item()
        loss_tr /= len(dl_tr)

        modelo.eval()
        loss_va = 0.0
        with torch.no_grad():
            for imgs, labels in dl_va:
                with torch.cuda.amp.autocast(enabled=usar_amp):
                    out = modelo(imgs.to(device))
                    loss_va += criterio(out, labels.to(device)).item()
        loss_va /= len(dl_va)

        hist_tr.append(loss_tr); hist_va.append(loss_va)

        if loss_va < mejor_val:
            mejor_val = loss_va
            mejor_pesos = {k: v.clone() for k, v in modelo.state_dict().items()}

        if (epoca+1) % cada_n_checkpoint == 0:
            ruta_ckpt = f"{carpeta_checkpoints}/{nombre}_epoca{epoca+1:03d}.pt"
            torch.save({"epoca": epoca+1, "state_dict": modelo.state_dict(),
                        "optim": optim.state_dict(), "loss_val": loss_va}, ruta_ckpt)

        if epoca % 5 == 0:
            print(f"  [{nombre}] Epoca {epoca:2d} | train: {loss_tr:.4f} | val: {loss_va:.4f}")

    modelo.load_state_dict(mejor_pesos)
    elapsed = time.time() - t0
    pico_vram = torch.cuda.max_memory_allocated()/1e6 if device.type == "cuda" else 0.0
    print(f"  [{nombre}] Mejor val: {mejor_val:.4f} | Tiempo: {elapsed/60:.1f} min | "
          f"Pico VRAM: {pico_vram:.0f} MB")
    return {"hist_tr": hist_tr, "hist_va": hist_va, "tiempo_s": elapsed, "pico_vram_mb": pico_vram}


In [ ]:
# --- Entrenamiento SIN precision mixta (linea base) ---
torch.manual_seed(SEMILLA)
modelo_fp32 = crear_resnet18()
resultado_fp32 = entrenar_con_checkpoint(
    modelo_fp32, dl_train_final, dl_val, nombre="resnet18_fp32",
    usar_amp=False, carpeta_checkpoints="checkpoints_fp32",
)


In [ ]:
# --- Entrenamiento CON precision mixta (AMP) ---
torch.manual_seed(SEMILLA)
modelo_amp = crear_resnet18()
resultado_amp = entrenar_con_checkpoint(
    modelo_amp, dl_train_final, dl_val, nombre="resnet18_amp",
    usar_amp=True, carpeta_checkpoints="checkpoints_amp",
)


In [ ]:
print("=== TABLA COMPARATIVA - FP32 vs AMP (FP16) ===")
print(f"{'':<10} {'Tiempo (min)':>14} {'Pico VRAM (MB)':>16} {'Mejor val loss':>16}")
print("-"*58)
for nombre, res in [("FP32", resultado_fp32), ("AMP", resultado_amp)]:
    print(f"{nombre:<10} {res['tiempo_s']/60:>14.2f} {res['pico_vram_mb']:>16.0f} "
          f"{min(res['hist_va']):>16.4f}")

ahorro_vram = (1 - resultado_amp['pico_vram_mb']/resultado_fp32['pico_vram_mb']) * 100
print(f"\nAhorro de VRAM con AMP: {ahorro_vram:.1f}%")
print("Nota: el ahorro teorico de activaciones en FP16 es ~50%, pero el ahorro")
print("total medido es menor porque el optimizador Adam mantiene los pesos")
print("maestros en FP32 para las actualizaciones de gradiente.")

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (nombre, res, color) in zip(axes, [
    ("FP32", resultado_fp32, "#4C72B0"),
    ("AMP (FP16)", resultado_amp, "#C44E52"),
]):
    ax.plot(res["hist_tr"], color=color, label="train")
    ax.plot(res["hist_va"], color=color, linestyle="--", label="val")
    ax.set_title(nombre); ax.set_xlabel("Epoca"); ax.set_ylabel("Cross-Entropy Loss")
    ax.legend(); ax.grid(alpha=0.3)
plt.suptitle("Curvas de perdida - FP32 vs precision mixta (AMP)")
plt.tight_layout(); plt.show()


### Manejo documentado de CUDA Out-Of-Memory

Se fuerza deliberadamente un `batch_size` demasiado grande para la VRAM
disponible, se captura la excepción, se libera la caché de CUDA y se reintenta
con un tamaño de batch reducido — replicando el flujo real de ajuste de
hiperparámetros bajo restricción de recursos ("ingeniería de trinchera").


In [ ]:
def intentar_batch(modelo, dl_tr, batch_size):
    try:
        imgs, labels = next(iter(DataLoader(ds_train_aug, batch_size=batch_size, shuffle=True)))
        imgs, labels = imgs.to(device), labels.to(device)
        with torch.cuda.amp.autocast():
            out = modelo(imgs)
            loss = nn.CrossEntropyLoss()(out, labels)
        loss.backward()
        print(f"OK  batch_size={batch_size} cabe en VRAM")
        return True
    except torch.cuda.OutOfMemoryError as e:
        print(f"CUDA OOM con batch_size={batch_size}: {str(e)[:120]}...")
        return False
    finally:
        modelo.zero_grad(set_to_none=True)

if device.type == "cuda":
    torch.cuda.empty_cache()
    BATCH_DEMASIADO_GRANDE = 8192
    print(f"Probando batch_size intencionalmente grande ({BATCH_DEMASIADO_GRANDE})...")
    exito = intentar_batch(modelo_amp, dl_train_final, BATCH_DEMASIADO_GRANDE)
    if not exito:
        torch.cuda.empty_cache()
        print(f"\nRecuperando: reduciendo a batch_size={BATCH_ENTRENAMIENTO}")
        exito_recuperado = intentar_batch(modelo_amp, dl_train_final, BATCH_ENTRENAMIENTO)
        print(f"Recuperado correctamente: {exito_recuperado}")
else:
    print("Sin GPU disponible: la demostracion de CUDA OOM requiere entorno con GPU (T4 en Colab).")


## 5. Optimización edge: cuantización post-entrenamiento INT8

Técnica de compresión elegida: **cuantización post-entrenamiento a INT8** vía
`tf.lite.TFLiteConverter`, usando un `representative_dataset` con imágenes
reales de entrenamiento (requisito de la cuantización INT8 completa: calibrar
los rangos de activación con datos reales, no aleatorios).

La latencia se mide en **CPU**, no en la T4: el beneficio de INT8 es una
historia de despliegue en el borde (tablet sin GPU), y medirla en GPU no
mostraría la ganancia real que importa para este caso de uso.


In [ ]:
# Exportación multi-formato del modelo entrenado
import os, torch, tensorflow as tf

os.makedirs("centinela_fase3_resnet18_tf", exist_ok=True)

# 1. Guardar .pth (ya se guarda durante el entrenamiento, verificar)
torch.save(modelo_amp.state_dict(), "centinela_fase3_resnet18.pth")
print("OK  centinela_fase3_resnet18.pth guardado")

# 2. Exportar a .onnx con opset 18 (compatible con PyTorch >= 2.0)
modelo_amp_cpu = modelo_amp.eval().cpu()
dummy_224 = torch.randn(1, 3, 224, 224)
torch.onnx.export(
    modelo_amp_cpu, dummy_224, "centinela_fase3_resnet18.onnx",
    input_names=["input"], output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=18,
)
print("OK  centinela_fase3_resnet18.onnx exportado (opset 18)")

# 3. Exportar a .keras directamente desde TensorFlow (sin onnx2tf)
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet",
    input_shape=(224, 224, 3), pooling="avg")
base.trainable = False
inp = tf.keras.Input(shape=(224, 224, 3))
x   = base(inp, training=False)
out = tf.keras.layers.Dense(3, activation="softmax")(x)
modelo_keras_real = tf.keras.Model(inp, out)
modelo_keras_real.save("centinela_fase3_resnet18_tf/centinela_fase3.keras")
print("OK  centinela_fase3.keras generado")
print("Contenido:", os.listdir("centinela_fase3_resnet18_tf"))


In [ ]:
# Conversión a TFLite float32 e INT8 desde .keras
import tensorflow as tf

# Cargar el modelo Keras guardado
modelo_keras_cargado = tf.keras.models.load_model(
    "centinela_fase3_resnet18_tf/centinela_fase3.keras")

# --- Conversión FLOAT (antes) ---
convertidor_float = tf.lite.TFLiteConverter.from_keras_model(modelo_keras_cargado)
tflite_float = convertidor_float.convert()
with open("centinela_fase3_resnet18_float.tflite", "wb") as f:
    f.write(tflite_float)
print(f"OK  float32 .tflite: {len(tflite_float)/1e6:.2f} MB")

# --- Conversión INT8 (después) ---
rutas_calibracion = rutas[:100]  # 100 imágenes reales de train para calibrar

def representative_dataset():
    for ruta in rutas_calibracion:
        img, _ = cargar_normalizar(ruta, 0)
        img = tf.expand_dims(img, axis=0)
        yield [img]

convertidor_int8 = tf.lite.TFLiteConverter.from_keras_model(modelo_keras_cargado)
convertidor_int8.optimizations = [tf.lite.Optimize.DEFAULT]
convertidor_int8.representative_dataset = representative_dataset
convertidor_int8.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
convertidor_int8.inference_input_type  = tf.int8
convertidor_int8.inference_output_type = tf.int8
tflite_int8 = convertidor_int8.convert()
with open("centinela_fase3_resnet18_int8.tflite", "wb") as f:
    f.write(tflite_int8)
print(f"OK  INT8    .tflite: {len(tflite_int8)/1e6:.2f} MB")
print(f"Reducción de tamaño: {(1 - len(tflite_int8)/len(tflite_float))*100:.1f}%")


In [ ]:
# --- Medir tamano, latencia (CPU) y accuracy/F1 en test para AMBAS versiones ---
def evaluar_tflite(ruta_tflite, es_int8):
    interprete = tf.lite.Interpreter(model_path=ruta_tflite, num_threads=1)
    interprete.allocate_tensors()
    entrada = interprete.get_input_details()[0]
    salida  = interprete.get_output_details()[0]

    # Guarda de forma: onnx2tf convierte NCHW (PyTorch) -> NHWC (TF). Si esto
    # no se cumple, el interprete NO va a fallar - va a devolver predicciones
    # silenciosamente incorrectas. Mejor fallar ruidoso aqui que confundir un
    # bug de layout con "la cuantizacion bajo mucho la accuracy".
    forma_esperada = [1, 224, 224, 3]
    assert list(entrada["shape"]) == forma_esperada, (
        f"Forma de entrada inesperada {list(entrada['shape'])}, se esperaba {forma_esperada}. "
        "Revisar la conversion onnx2tf (posible layout NCHW sin transponer)."
    )

    rutas_test, ys_test = [], []
    for c, nombre_clase in enumerate(nombres_clases):
        for p in (RAIZ_IMG / "test" / nombre_clase).glob("*.png"):
            rutas_test.append(str(p)); ys_test.append(c)

    preds, tiempos = [], []
    for ruta, y in zip(rutas_test, ys_test):
        img, _ = cargar_normalizar(ruta, y)
        img = tf.expand_dims(img, axis=0)
        if es_int8:
            escala, punto_cero = entrada["quantization"]
            img = tf.cast(img / escala + punto_cero, tf.int8)
        t0 = time.perf_counter()
        interprete.set_tensor(entrada["index"], img)
        interprete.invoke()
        tiempos.append(time.perf_counter() - t0)
        out = interprete.get_tensor(salida["index"])
        preds.append(int(np.argmax(out)))

    tam_mb = os.path.getsize(ruta_tflite) / 1e6
    latencia_ms = np.mean(tiempos) * 1000
    acc = accuracy_score(ys_test, preds)
    f1  = f1_score(ys_test, preds, average="macro", zero_division=0)
    return {"tam_mb": tam_mb, "latencia_ms": latencia_ms, "accuracy": acc,
            "f1_macro": f1, "preds": preds, "ys_test": ys_test, "rutas_test": rutas_test}

metricas_float = evaluar_tflite("centinela_fase3_resnet18_float.tflite", es_int8=False)
metricas_int8  = evaluar_tflite("centinela_fase3_resnet18_int8.tflite",  es_int8=True)

print("=== TABLA COMPARATIVA - Optimizacion Edge (antes / despues) ===")
print(f"{'Metrica':<24} {'Antes (float32)':>18} {'Despues (INT8)':>18}")
print("-"*62)
print(f"{'Tamano modelo (MB)':<24} {metricas_float['tam_mb']:>18.2f} {metricas_int8['tam_mb']:>18.2f}")
print(f"{'Latencia CPU (ms/img)':<24} {metricas_float['latencia_ms']:>18.2f} {metricas_int8['latencia_ms']:>18.2f}")
print(f"{'Accuracy (test)':<24} {metricas_float['accuracy']:>18.3f} {metricas_int8['accuracy']:>18.3f}")
print(f"{'F1 macro (test)':<24} {metricas_float['f1_macro']:>18.3f} {metricas_int8['f1_macro']:>18.3f}")


### Verificación de fidelidad de la conversión (no solo accuracy vs. etiqueta real)

Que el `.tflite` tenga baja accuracy contra la etiqueta real no dice *por qué*:
puede ser que el modelo en sí sea débil (la Fase 2 ya reportó ~50% de accuracy
en esta tarea), o puede ser que la conversión ONNX -> TF haya introducido un
bug de layout silencioso. La forma de distinguirlo es comparar el `.tflite`
**contra las predicciones de PyTorch en las mismas imágenes**, no solo contra
la etiqueta real: si coinciden en la gran mayoría de los casos, la conversión
es fiel y cualquier caída de accuracy es del modelo, no del export.


In [ ]:
modelo_amp_cpu.eval()
preds_pytorch = []
with torch.no_grad():
    for ruta in metricas_float["rutas_test"]:
        img = transform_eval(Image.open(ruta).convert("RGB")).unsqueeze(0)
        preds_pytorch.append(int(modelo_amp_cpu(img).argmax(dim=1).item()))

acuerdo_float = np.mean(np.array(preds_pytorch) == np.array(metricas_float["preds"]))
acuerdo_int8  = np.mean(np.array(preds_pytorch) == np.array(metricas_int8["preds"]))

print(f"Acuerdo PyTorch vs TFLite-float: {acuerdo_float*100:.1f}%  (esperado: cercano a 100%)")
print(f"Acuerdo PyTorch vs TFLite-INT8 : {acuerdo_int8*100:.1f}%  (algo menor es normal por la cuantizacion)")
if acuerdo_float < 0.9:
    print("\nALERTA: el acuerdo float es bajo. Esto apunta a un problema de la")
    print("conversion (layout NCHW/NHWC, normalizacion distinta, etc.), NO a que")
    print("el modelo sea malo. Revisar antes de reportar estos numeros en el informe.")


**Discusión del trade-off:** completar tras ejecutar la celda anterior con
los números reales. Guía de lectura: si la caída de accuracy/F1 de INT8 frente
a float32 es pequeña (unos pocos puntos porcentuales) frente a la reducción de
tamaño/latencia, la cuantización es aceptable para un despliegue de campo donde
la conectividad es intermitente y el modelo debe caber y correr rápido en una
tablet de gama media. Si la caída es grande, documentar que se prefiere el
modelo float32 en el borde a costa de más espacio/latencia, o explorar poda o
destilación como alternativa.


## 6. Exportación final, verificación de consistencia y ruta de despliegue

### 6.1 Rama A (visual) — los 3 formatos y verificación ONNX

Ya se generaron a lo largo de la Sección 5: `.onnx` y el SavedModel/Keras
(`.keras` dentro de `centinela_fase3_resnet18_tf/`). Aquí se guarda también el
`.pth` final y se corre la verificación numérica `np.allclose` con imágenes
**reales** de test (no el tensor aleatorio de la prueba de humo).


In [ ]:
torch.save(modelo_amp_cpu.state_dict(), "centinela_fase3_resnet18.pth")
print("OK  centinela_fase3_resnet18.pth guardado")

ruta_keras = [p for p in Path("centinela_fase3_resnet18_tf").glob("*.keras")]
if ruta_keras:
    print("OK  Modelo .keras encontrado:", ruta_keras[0])
else:
    print("AVISO: onnx2tf no genero un .keras explicito en esta version; "
          "el SavedModel en centinela_fase3_resnet18_tf/ es equivalente y "
          "se documenta como tal en el informe.")


In [ ]:
# Verificacion PyTorch vs ONNX Runtime con imagenes REALES de test (no aleatorias)
import onnxruntime as ort

modelo_amp_cpu.eval()  # de nuevo eval() explicito antes de comparar - critico con BatchNorm
sesion = ort.InferenceSession("centinela_fase3_resnet18.onnx")

lote_test, _ = next(iter(dl_test))
with torch.no_grad():
    salida_pt = modelo_amp_cpu(lote_test).numpy()

salida_onnx = sesion.run(None, {"input": lote_test.numpy()})[0]

coinciden = np.allclose(salida_pt, salida_onnx, atol=1e-4)
print(f"Verificacion PyTorch vs ONNX Runtime (atol=1e-4) sobre batch real de test: {coinciden}")
diferencia_max = np.abs(salida_pt - salida_onnx).max()
print(f"Diferencia absoluta maxima observada: {diferencia_max:.2e}")


### 6.2 Rama B (temporal) — export a ONNX y ruta de servicio por API REST

La rama temporal (RNN/LSTM/GRU, Fase 2 Rama B) **no** se lleva a LiteRT: la
exportación de capas recurrentes (`nn.GRU`/`nn.LSTM`) a TFLite es notoriamente
poco confiable (operadores no siempre soportados de forma nativa en LiteRT).
En su lugar se exporta a ONNX — que sí soporta operadores recurrentes de forma
estable — y se sirve detrás de un endpoint REST con ONNX Runtime, consistente
con que el escenario de campo (tablet, conectividad intermitente) es
precisamente donde la rama visual offline importa más; la rama temporal opera
con datos de mercado (TRM) que de todas formas requieren conectividad para
actualizarse, así que depender de una API en la nube no añade una restricción
nueva.


In [ ]:
class ModeloRecurrente(nn.Module):
    def __init__(self, tipo="GRU", input_size=1, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.tipo = tipo
        cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[tipo]
        self.rnn = cls(input_size, hidden_size, num_layers, batch_first=True, dropout=dropout)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

# Reentrenar rapido el GRU (o cargar centinela_fase2_gru.pt si esta disponible en el entorno)
vals = df["retorno"].values.reshape(-1, 1)
n = len(vals)
i_tr, i_va = int(n*0.70), int(n*0.85)
scaler_gru = MinMaxScaler()
train_sc = scaler_gru.fit_transform(vals[:i_tr])

def crear_ventanas(serie, window):
    X, y = [], []
    for i in range(window, len(serie)):
        X.append(serie[i-window:i, 0]); y.append(serie[i, 0])
    return np.array(X), np.array(y)

X_tr, y_tr = crear_ventanas(train_sc, WINDOW)
X_tr_t = torch.tensor(X_tr, dtype=torch.float32).unsqueeze(-1)
y_tr_t = torch.tensor(y_tr, dtype=torch.float32).view(-1, 1)

torch.manual_seed(SEMILLA)
modelo_gru = ModeloRecurrente(tipo="GRU")
try:
    modelo_gru.load_state_dict(torch.load("centinela_fase2_gru.pt", map_location="cpu"))
    print("OK  Pesos de Fase 2 (centinela_fase2_gru.pt) cargados")
except FileNotFoundError:
    print("centinela_fase2_gru.pt no esta en este entorno - entrenando rapido para tener pesos validos")
    opt = torch.optim.Adam(modelo_gru.parameters(), lr=1e-3)
    crit = nn.MSELoss()
    for _ in range(30):
        modelo_gru.train(); opt.zero_grad()
        loss = crit(modelo_gru(X_tr_t), y_tr_t)
        loss.backward(); opt.step()
    print(f"Entrenamiento rapido completado, loss final: {loss.item():.6f}")

modelo_gru.eval()
dummy_secuencia = torch.randn(1, WINDOW, 1)
torch.onnx.export(
    modelo_gru, dummy_secuencia, "centinela_fase3_gru.onnx",
    input_names=["secuencia"], output_names=["retorno_predicho"],
    dynamic_axes={"secuencia": {0: "batch"}, "retorno_predicho": {0: "batch"}},
    opset_version=18,
)

sesion_gru = ort.InferenceSession("centinela_fase3_gru.onnx")
with torch.no_grad():
    salida_pt_gru = modelo_gru(dummy_secuencia).numpy()
salida_onnx_gru = sesion_gru.run(None, {"secuencia": dummy_secuencia.numpy()})[0]
print("Verificacion GRU PyTorch vs ONNX (atol=1e-4):",
      np.allclose(salida_pt_gru, salida_onnx_gru, atol=1e-4))


In [ ]:
%%writefile app_rama_b.py
"""
Esqueleto de servicio REST para la rama temporal (GRU) del Proyecto Centinela.
No se despliega en este notebook; documenta la ruta de servicio en la nube
para el endpoint que consumiria la tablet cuando tenga conectividad.

Ejecutar localmente con: uvicorn app_rama_b:app --host 0.0.0.0 --port 8000
"""
import numpy as np
import onnxruntime as ort
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Centinela - Rama B (temporal)")
sesion = ort.InferenceSession("centinela_fase3_gru.onnx")

class VentanaRetornos(BaseModel):
    retornos_escalados: list[float]  # 30 valores, ya escalados con el MinMaxScaler de train

@app.post("/predecir_retorno")
def predecir_retorno(datos: VentanaRetornos):
    secuencia = np.array(datos.retornos_escalados, dtype=np.float32).reshape(1, -1, 1)
    salida = sesion.run(None, {"secuencia": secuencia})[0]
    return {"retorno_predicho_escalado": float(salida[0][0])}


### 6.3 Ruta de despliegue completa

| | Rama A (visual) | Rama B (temporal) |
|---|---|---|
| **Modelo** | ResNet18, Feature Extraction | GRU (2 capas, 64 unidades) |
| **Formato de despliegue** | `.tflite` INT8 (LiteRT) | `.onnx` |
| **Dónde corre** | Offline, en el dispositivo (tablet) | Servidor en la nube, vía API REST |
| **Runtime** | LiteRT (TensorFlow Lite) | ONNX Runtime detrás de FastAPI |
| **Por qué esta división** | Debe funcionar sin conectividad; INT8 reduce tamaño/latencia para hardware de tablet | Exportación de capas recurrentes a LiteRT no es confiable; los datos de TRM que consume ya requieren conexión para actualizarse |

**Principios de IA responsable aplicados:**
- **Transparencia:** el endpoint y la app de tablet deben mostrar la
  probabilidad/confianza de la predicción junto a la clase, no solo la
  etiqueta — un usuario de campo necesita saber cuándo el modelo está inseguro.
- **Beneficio social:** el sistema busca anticipar presión sobre el poder
  adquisitivo del SMLV/TRM para que hogares y pequeños negocios puedan
  anticipar decisiones, no sustituir asesoría financiera formal.
- **Mitigación de falsos negativos** (no alertar una depreciación real): el
  umbral de alerta debe calibrarse de forma conservadora — es preferible una
  alerta de más (revisable por un humano) que una omitida.
- **Mitigación de falsos positivos** (alertar sin motivo real): alertas
  repetidas sin fundamento erosionan la confianza en el sistema ("fatiga de
  alerta"); se recomienda mostrar siempre la confianza del modelo y evitar
  notificaciones automáticas sin revisión humana para decisiones de alto impacto.


## 7. Resumen de la Fase 3

**Qué se industrializó:** las Ramas A (visual, ResNet18 sobre matrices de
recurrencia de la TRM) y B (temporal, GRU sobre retornos diarios) de la Fase 2
del Proyecto Centinela, siguiendo el reto de MLOps del Módulo 3 (adaptado desde
el escenario de referencia de hojas de café a los datos propios del proyecto).

**Pipeline:** mismo pipeline lógico implementado en `tf.data` y `DataLoader`
sobre imágenes materializadas en disco (`imagenes/{train,val,test}/<clase>/`),
con comparación de tiempo de carga por época.

**Augmentation:** 4 transformaciones justificadas (rotación, flip horizontal,
brillo, recorte redimensionado), comparación de accuracy con/sin augmentation.

**GPU:** ResNet18 con precisión mixta (AMP/FP16), checkpointing cada 5 épocas,
manejo documentado de CUDA OOM.

**Edge:** cuantización post-entrenamiento INT8, tabla antes/después de
tamaño, latencia (CPU) y accuracy/F1.

**Exportación:** `.pth`, `.onnx`, `.keras`/SavedModel para la rama visual;
`.onnx` para la rama temporal; verificación numérica PyTorch vs. ONNX Runtime
con `np.allclose(atol=1e-4)`.

**Ruta de despliegue:** rama visual offline en tablet vía LiteRT; rama
temporal servida en la nube vía API REST (esqueleto FastAPI + ONNX Runtime).

**Limitación honesta:** la cadena de conversión ONNX -> Keras -> LiteRT es
sensible a versiones de librerías; el notebook incluye una prueba de humo
(Sección 0.5) y una ruta alterna (`ai-edge-torch`) precisamente porque esta
dependencia no se puede validar fuera de un entorno Colab real.
